# 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("All libraries imported successfully")

# 2. Loading Clean Data

Dataset: e-store customer data. Target (`Exited`) = 1 if customer will buy products, 0 if not.

In [ ]:
df = pd.read_csv("./data/clean_data_v1.csv", index_col=0)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited']

print(f"Features: {list(X.columns)}")
print(f"Target distribution (1=will buy, 0=won't buy):\n{y.value_counts()}")

# 3. Splitting Data

Split into training (60%), validation (20%), and testing (20%).
We use stratification to preserve the class imbalance ratio across splits.

In [ ]:
# First split: train (60%) vs temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Second split: validation (20%) vs test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print()
print(f"Train - buyers ratio: {y_train.mean():.3f}")
print(f"Val   - buyers ratio: {y_val.mean():.3f}")
print(f"Test  - buyers ratio: {y_test.mean():.3f}")

# 4. Feature Scaling

Scale features so models like Logistic Regression and SVM perform well.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features scaled (mean=0, std=1)")

# 5. Building and Training Models

Four classifiers to predict if a customer will buy products from the e-store.
We use `class_weight='balanced'` to handle the imbalanced dataset without discarding data.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100),
    "SVM": SVC(class_weight='balanced', random_state=42, probability=True),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42, max_depth=10)
}

trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    train_acc = model.score(X_train_scaled, y_train)
    val_acc = model.score(X_val_scaled, y_val)
    print(f"  Train accuracy: {train_acc:.4f}")
    print(f"  Val accuracy:   {val_acc:.4f}")
    print()

print("All models trained successfully")

# 6. Validation Prediction Probabilities

Generate the probability that a customer **will buy** for each model.

In [ ]:
val_probs = {}
val_preds = {}

for name, model in trained_models.items():
    probs = model.predict_proba(X_val_scaled)[:, 1]
    preds = model.predict(X_val_scaled)
    val_probs[name] = probs
    val_preds[name] = preds
    print(f"{name}: {len(probs)} predictions generated")

# 7. Evaluating at Multiple Thresholds

We test thresholds from 0.1 to 0.9 to find the best cut-off for each model.
Focus on **F1 score** since the data is imbalanced.

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)

results = []

for name in trained_models:
    probs = val_probs[name]
    for t in thresholds:
        preds_t = (probs >= t).astype(int)
        results.append({
            'Model': name,
            'Threshold': round(t, 1),
            'Accuracy': accuracy_score(y_val, preds_t),
            'Precision': precision_score(y_val, preds_t),
            'Recall': recall_score(y_val, preds_t),
            'F1': f1_score(y_val, preds_t)
        })

df_results = pd.DataFrame(results)
df_results_pivot = df_results.pivot_table(index='Threshold', columns='Model', values='F1')
print("F1 Score by Model and Threshold:")
df_results_pivot

# 8. Default vs Optimal Threshold

Default = 0.5. Optimal = threshold that maximizes F1 score.
This helps us tune the model to better identify customers who **will buy**.

In [ ]:
threshold_comparison = []

for name in trained_models:
    probs = val_probs[name]
    
    # Default (0.5)
    preds_default = (probs >= 0.5).astype(int)
    f1_default = f1_score(y_val, preds_default)
    
    # Optimal
    best_f1 = 0
    best_t = 0.5
    for t in thresholds:
        preds_t = (probs >= t).astype(int)
        f = f1_score(y_val, preds_t)
        if f > best_f1:
            best_f1 = f
            best_t = t
    
    threshold_comparison.append({
        'Model': name,
        'Default F1 (0.5)': round(f1_default, 4),
        'Optimal Threshold': best_t,
        'Optimal F1': round(best_f1, 4),
        'Improvement': round(best_f1 - f1_default, 4)
    })

df_comparison = pd.DataFrame(threshold_comparison)
df_comparison

# 9. Classification Report and Confusion Matrix (Optimal Threshold)

Labels: `Won't Buy` (0) vs `Will Buy` (1).

In [ ]:
target_labels = ["Won't Buy", "Will Buy"]

for name in trained_models:
    probs = val_probs[name]
    
    # Find optimal threshold
    best_t = 0.5
    best_f1 = 0
    for t in thresholds:
        preds_t = (probs >= t).astype(int)
        f = f1_score(y_val, preds_t)
        if f > best_f1:
            best_f1 = f
            best_t = t
    
    preds_opt = (probs >= best_t).astype(int)
    
    print(f"\n{'='*55}")
    print(f"{name} (Threshold = {best_t})")
    print('='*55)
    print("\nClassification Report:")
    print(classification_report(y_val, preds_opt, target_names=target_labels))
    
    cm = confusion_matrix(y_val, preds_opt)
    print("Confusion Matrix:")
    print(f"                  Predicted")
    print(f"             Won't Buy   Will Buy")
    print(f"Actual Won't Buy   {cm[0,0]:>5d}        {cm[0,1]:>5d}")
    print(f"       Will Buy     {cm[1,0]:>5d}        {cm[1,1]:>5d}")

# 10. Precision-Recall Curve

Higher curves = better at identifying customers who will buy.

In [ ]:
plt.figure(figsize=(10, 6))

for name in trained_models:
    precision, recall, _ = precision_recall_curve(y_val, val_probs[name])
    plt.plot(recall, precision, label=name, linewidth=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Predicting Purchase)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 11. ROC Curve

AUC closer to 1.0 = better at separating buyers from non-buyers.

In [ ]:
plt.figure(figsize=(10, 6))

for name in trained_models:
    fpr, tpr, _ = roc_curve(y_val, val_probs[name])
    auc = roc_auc_score(y_val, val_probs[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Purchase Prediction)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 12. Test Set Evaluation (Best Model)

Pick the model with the highest F1 on validation and evaluate on the held-out test set.
This gives us a realistic estimate of how the model will perform on new e-store customers.

In [ ]:
# Find best model by validation F1
best_model_name = None
best_f1 = 0

for name in trained_models:
    for t in thresholds:
        preds_t = (val_probs[name] >= t).astype(int)
        f = f1_score(y_val, preds_t)
        if f > best_f1:
            best_f1 = f
            best_model_name = name
            best_t = t

print(f"Best model on validation: {best_model_name}")
print(f"Optimal threshold: {best_t}")
print(f"Validation F1: {best_f1:.4f}")
print()

# Evaluate on test set
model = trained_models[best_model_name]
test_probs = model.predict_proba(X_test_scaled)[:, 1]
test_preds = (test_probs >= best_t).astype(int)

print("Test Set Performance:")
print(f"Accuracy:  {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall:    {recall_score(y_test, test_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, test_preds):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, test_preds, target_names=target_labels))

cm_test = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_labels,
            yticklabels=target_labels)
plt.title(f'Confusion Matrix - {best_model_name} (Test Set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()